# MTPL frequency pricing model

This notebook is the complete analyst workflow: settings, SQL, feature transforms, model definition, fitting, publication, and optional review/deployment. Generated audit identifiers and SQL plumbing stay behind the helper functions.

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import text
from superglm import Categorical, Numeric, Spline, SuperGLM

from pricing_pipeline.infra.schema import schema_names_from_connectable
from pricing_pipeline.models.config import ValidationSplitConfig
from pricing_pipeline.notebook import (
    PricingModelSpec,
    build_candidate,
    connect,
    deploy_package,
    open_candidate,
    publish_candidate,
    publish_edits,
    register_model,
)

## Analyst settings

These are the important model and data decisions. `DATA_AS_OF` is the source-data cutoff represented by this frame; it is not a deployment date.

In [ ]:
MODEL_DIR = Path("pricing_models/mtpl_frequency")
if not MODEL_DIR.is_dir():
    MODEL_DIR = Path.cwd()
MODEL_DIR = MODEL_DIR.resolve()

FEATURE_COLUMNS = (
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "LogDensity",
    "Area",
    "VehPower",
    "VehBrand",
    "VehGas",
    "Region",
)

MODEL = PricingModelSpec(
    name="MTPL_FREQ",
    label="Motor frequency",
    target="ClaimNb",
    model_type="superglm_poisson",
    deployment_slot="MTPL_FREQ_UAT",
    features=FEATURE_COLUMNS,
    dataset_name="freMTPL2freq_model_frame",
    source_system="freMTPL_raw_sql",
    pk_columns=("IDpol",),
    exposure_column="Exposure",
    validation=ValidationSplitConfig.kfold(
        n_splits=5,
        random_state=42,
        shuffle=True,
        materialize=True,
    ),
)

DATA_AS_OF = date(2026, 6, 30)

## Optional actions

Candidate publication does not deploy anything. Editor and deployment actions stay off unless deliberately enabled.

In [ ]:
RUN_EDITOR = False
EDIT_REASON = ""

DEPLOY = False
DEPLOYMENT_REASON = ""

## Connect and register the stable model identity

Registration is idempotent. SQL owns `model_id`; reruns validate the stored identity.

In [ ]:
pricing = connect()
model = register_model(
    pricing,
    MODEL,
    source_root=MODEL_DIR,
)
display({"Model": model.name, "SQL model ID": model.model_id})

## Read source data from SQL

Keep the primary key and every column needed for transforms, fitting, and export.

In [ ]:
schemas = schema_names_from_connectable(pricing.engine)
SOURCE_SQL = f"""
SELECT
    IDpol, ClaimNb, Exposure, Area, VehPower, VehAge, DrivAge,
    BonusMalus, VehBrand, VehGas, Density, Region
FROM {schemas.pricing}.FREMTPL_RAW
ORDER BY IDpol
"""
raw = pd.read_sql_query(text(SOURCE_SQL), pricing.engine)
if raw.empty:
    raise RuntimeError("FREMTPL_RAW is empty; load source rows before modelling.")
raw.head()

## Build the final model frame

Feature transforms remain ordinary, visible Python. The helper derives `X`, `y`, row identity, the exposure offset, and audit metadata from `MODEL`.

In [ ]:
frame = raw.loc[raw["Exposure"].astype(float) > 0].copy()
frame["LogDensity"] = np.log(frame["Density"].astype(float).clip(lower=1.0))
frame = frame.loc[
    :, ["IDpol", "ClaimNb", "Exposure", *FEATURE_COLUMNS]
].sort_values("IDpol").reset_index(drop=True)
frame.head()

## Define the model

Feature types and model choices stay ordinary Python.

In [ ]:
def make_model():
    return SuperGLM(
        family="poisson",
        selection_penalty=0.0,
        discrete=True,
        n_bins=256,
        features={
            "VehAge": Spline(),
            "DrivAge": Spline(),
            "BonusMalus": Spline(),
            "LogDensity": Numeric(),
            "Area": Categorical(),
            "VehPower": Categorical(),
            "VehBrand": Categorical(),
            "VehGas": Categorical(),
            "Region": Categorical(),
        },
    )

## Fit and capture audit evidence

The analyst supplies the model frame and its data cutoff. Generated manifests, folds, hashes, versions, and offset-export metadata are plumbing.

In [ ]:
candidate = build_candidate(
    pricing,
    model=model,
    frame=frame,
    model_factory=make_model,
    data_as_of=DATA_AS_OF,
)
candidate.metrics

## Publish the immutable candidate

This writes the audit lineage and rating tables to SQL. It does not create a deployment or invent an effective date.

In [ ]:
published = publish_candidate(pricing, candidate)
display({
    "SQL model ID": published.model_id,
    "Dataset manifest": published.manifest_id,
    "Validation split": published.split_set_id,
    "Model run": published.model_run_id,
    "Rate package": published.rate_package_id,
    "Package version": published.package_version,
    "State": published.package_status,
})

## Optional market edit

Enable `RUN_EDITOR` in the top action cell, make changes in the editor, provide a reason, and publish an immutable child package.

In [ ]:
editable = None
if RUN_EDITOR:
    editable = open_candidate(
        pricing, model=model, package_version=published.package_version
    )
    display(editable.editor())

In [ ]:
edited = None
if RUN_EDITOR:
    if not EDIT_REASON.strip():
        raise ValueError("Describe the market or underwriting reason for the edit.")
    edited = publish_edits(
        pricing,
        model=model,
        candidate=editable,
        reason=EDIT_REASON,
    )
    display({
        "Edited model run": edited.model_run_id,
        "Edited rate package": edited.rate_package_id,
        "Edited package version": edited.package_version,
    })

## Optional deployment

Deployment is a separate deliberate action. SQL records its actual activation timestamp.

In [ ]:
if DEPLOY:
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the live package.")
    package_to_deploy = edited if edited is not None else published
    deployment = deploy_package(
        pricing,
        model=model,
        package=package_to_deploy,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)